In [1]:
# ==============================================================================
# CLUSTERING PIPELINE — SCENARIO 2 — Pure Hamming, booleans only, no weights
# ==============================================================================


# ==============================================================================
# 0. CONFIGURATION
# ==============================================================================

CSV_PATH   = "df_final_binaire_imputed.csv"
OUTPUT_DIR = "Results/Regular_clustering/Full_dataset/Without_counts/s2_noweights_boolonly"

GPU_DEVICE_ID = 1

UMAP_N_NEIGHBORS  = 30
UMAP_MIN_DIST     = 0.0
UMAP_N_COMPONENTS = 10
UMAP_RANDOM_STATE = 42

VIZ_N_NEIGHBORS = 30
VIZ_MIN_DIST    = 0.1
VIZ_TSNE_PERP   = 30
VIZ_TSNE_ITER   = 1500

HDBSCAN_MIN_CLUSTER_SIZE = 1000
HDBSCAN_MIN_SAMPLES      = 10
HDBSCAN_CLUSTER_METHOD   = "eom"

SWEEP_VALUES = list(range(500, 2200, 100))

W_SILHOUETTE = 0.4
W_STABILITY  = 0.3
W_OUTLIER    = 0.3


# ==============================================================================
# 1. IMPORTS
# ==============================================================================

import os
import logging
import time
import hdbscan
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import torch
import umap
import umap.umap_ as umap_reduce
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

torch.cuda.set_device(GPU_DEVICE_ID)
torch.cuda.set_per_process_memory_fraction(0.5, device=GPU_DEVICE_ID)
print(f"GPU : {torch.cuda.get_device_name(GPU_DEVICE_ID)}")
print(f"Available memory : {torch.cuda.get_device_properties(GPU_DEVICE_ID).total_memory / 1e9:.0f} GB")


# ==============================================================================
# 2. COLUMN DEFINITIONS
# ==============================================================================

IMAGING_COLS_BOOL = {
    "has_ultrasound", "has_ct_scan", "has_xray", "has_mri",
    # "has_radio_interventional", "has_nuclear_medicine" → too rare, removed
}

BIO_EXAMS = {
    "has_blood_test", "has_culture",
    "has_lumbar_puncture", "has_blood_gas",
}

PROCEDURE_COLS   = {"had_ekg"}

DISPOSITION_COLS = {
    "hospitalization", "observation_unit", "inter_facility_transfer",
}

# All features — booleans only, no counts
ALL_FEATURES = list(IMAGING_COLS_BOOL | BIO_EXAMS | PROCEDURE_COLS | DISPOSITION_COLS)

SCENARIOS = {
    "scenario_2": ALL_FEATURES
}


# ==============================================================================
# 3. DATA LOADING
# ==============================================================================

def load_data(csv_path: str = CSV_PATH) -> pd.DataFrame:
    df = pd.read_csv(csv_path, low_memory=False)
    print(f"Dataset loaded : {len(df)} rows, {df.shape[1]} columns")
    return df


# ==============================================================================
# 4. DISTANCE MATRIX — GPU (pure Hamming, booleans only)
# ==============================================================================

def compute_distance_matrix_gpu(
    df_sub:    pd.DataFrame,
    binary_cols: list,
    device_id: int = GPU_DEVICE_ID,
) -> np.ndarray:
    """
    Pure Hamming distance matrix on GPU.
    No Manhattan, no StandardScaler, no weights.
    """
    device = torch.device(f"cuda:{device_id}")
    n      = len(df_sub)
    D      = torch.zeros((n, n), device=device, dtype=torch.float32)

    for col in binary_cols:
        vals = torch.tensor(
            df_sub[col].values, dtype=torch.float32, device=device
        ).unsqueeze(1)
        D += torch.cdist(vals, vals, p=1)

    D.fill_diagonal_(0)
    return D.cpu().numpy().astype(np.float64)


# ==============================================================================
# 5. CLUSTERING — run_hdbscan
# ==============================================================================

def run_hdbscan(
    df:               pd.DataFrame,
    run_label:        str,
    min_cluster_size: int = HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples:      int = HDBSCAN_MIN_SAMPLES,
):
    out_dir = os.path.join(OUTPUT_DIR, run_label)
    os.makedirs(out_dir, exist_ok=True)

    # ── Select and clean columns ───────────────────────────────────────────────
    cols   = [c for c in ALL_FEATURES if c in df.columns]
    df_sub = df[cols].dropna().copy()
    idx    = df_sub.index

    # ── All columns are binary → detect automatically ──────────────────────────
    binary_cols = [
        c for c in df_sub.columns
        if set(df_sub[c].dropna().unique()) <= {0, 1}
    ]

    log.info(f"[{run_label}] binary cols = {len(binary_cols)} | n = {len(df_sub)}")

    # ── Distance matrix — pure Hamming ─────────────────────────────────────────
    t0 = time.time()
    D  = compute_distance_matrix_gpu(df_sub, binary_cols=binary_cols)
    log.info(f"[{run_label}] Distance matrix : {time.time()-t0:.1f}s")

    # ── HDBSCAN ────────────────────────────────────────────────────────────────
    t0 = time.time()
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size         = min_cluster_size,
        min_samples              = min_samples,
        metric                   = "precomputed",
        cluster_selection_method = HDBSCAN_CLUSTER_METHOD,
        gen_min_span_tree        = True,
    ).fit(D)
    log.info(f"[{run_label}] HDBSCAN : {time.time()-t0:.1f}s")

    labels     = clusterer.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()

    log.info(
        f"[{run_label}] clusters={n_clusters} | "
        f"noise={n_noise} ({100*n_noise/len(labels):.1f}%)"
    )

    # ── Export CSV ─────────────────────────────────────────────────────────────
    df_out = df.loc[idx].copy()
    df_out["cluster"] = labels
    df_out.to_csv(os.path.join(out_dir, f"clustering_mcs{min_cluster_size}.csv"), index=False)
    log.info(f"[{run_label}] CSV exported.")

    return df_sub, D, labels, clusterer


# ==============================================================================
# 6. SWEEP
# ==============================================================================

def run_hdbscan_sweep(D: np.ndarray, sweep_values=SWEEP_VALUES) -> pd.DataFrame:
    np.fill_diagonal(D, 0)
    results = []

    for mcs in sweep_values:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size         = mcs,
            min_samples              = HDBSCAN_MIN_SAMPLES,
            metric                   = "precomputed",
            cluster_selection_method = HDBSCAN_CLUSTER_METHOD,
        ).fit(D)

        labels          = clusterer.labels_
        unique_clusters = np.unique(labels[labels >= 0])
        n_clusters      = len(unique_clusters)

        sil = (
            silhouette_score(D, labels, metric="precomputed")
            if n_clusters >= 2 else np.nan
        )
        stability = (
            float(np.mean(clusterer.cluster_persistence_))
            if len(clusterer.cluster_persistence_) > 0 else np.nan
        )

        results.append({
            "min_cluster_size": mcs,
            "n_clusters":       n_clusters,
            "silhouette":       sil,
            "outlier_rate":     float(np.mean(labels == -1)),
            "stability":        stability,
            "labels":           labels,
            "probabilities":    clusterer.probabilities_,
            "cluster_sizes":    {c: int(np.sum(labels == c)) for c in unique_clusters},
        })

    return pd.DataFrame(results)


def compute_combined_score(df_sweep: pd.DataFrame) -> pd.DataFrame:
    df = df_sweep.copy()
    df["silhouette"] = df["silhouette"].fillna(0)
    df["stability"]  = df["stability"].fillna(0)

    def _minmax(s):
        mn, mx = s.min(), s.max()
        return (s - mn) / (mx - mn) if mx != mn else s * 0

    df["combined_score"] = (
        W_SILHOUETTE * _minmax(df["silhouette"])
        + W_STABILITY  * _minmax(df["stability"])
        - W_OUTLIER    * df["outlier_rate"]
    )
    return df


# ==============================================================================
# 7. VISUALISATION — SWEEP CURVES
# ==============================================================================

def plot_sweep_curves(df_sweep, title, out_dir, filename_prefix):
    os.makedirs(out_dir, exist_ok=True)
    x = df_sweep["min_cluster_size"]

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(x, df_sweep["silhouette"], color="tab:blue", marker="o", label="Silhouette")
    ax1.set_xlabel("min_cluster_size")
    ax1.set_ylabel("Silhouette score", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")
    ax2 = ax1.twinx()
    ax2.plot(x, df_sweep["stability"], color="tab:red", marker="s", label="Stability")
    ax2.set_ylabel("Mean stability", color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")
    plt.title(title)
    fig.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{filename_prefix}_silhouette_stability.png"),
                dpi=150, facecolor="white")
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(x, df_sweep["combined_score"], color="tab:green", marker="d", linewidth=2)
    plt.xlabel("min_cluster_size")
    plt.ylabel("Combined score")
    plt.title("Combined score (silhouette + stability - outliers)")
    plt.grid(True)
    plt.savefig(os.path.join(out_dir, f"{filename_prefix}_combined_score.png"),
                dpi=150, facecolor="white")
    plt.close()
    log.info(f"[{filename_prefix}] Sweep curves saved.")


# ==============================================================================
# 8. VISUALISATION — CLUSTER PROFILES
# ==============================================================================

def plot_cluster_heatmap(df_sub, labels, title, out_dir, filename):
    os.makedirs(out_dir, exist_ok=True)
    df_prof            = df_sub.copy()
    df_prof["cluster"] = labels
    df_prof            = df_prof[df_prof["cluster"] != -1]

    numeric_cols = [
        c for c in df_prof.select_dtypes(include=["number"]).columns
        if c != "cluster"
    ]
    profile = df_prof[numeric_cols + ["cluster"]].groupby("cluster").mean().round(3)

    n_cols = profile.shape[1]
    n_rows = profile.shape[0]

    plt.figure(figsize=(max(8, n_cols * 0.8), max(4, n_rows * 0.6)))
    sns.heatmap(profile, annot=True, cmap="YlOrRd", fmt=".2f",
                annot_kws={"size": max(6, min(10, 80 // n_cols))})
    plt.title(title)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, filename), dpi=150,
                bbox_inches="tight", facecolor="white")
    plt.close()
    log.info(f"Heatmap saved.")
    return profile


def plot_cluster_dendrogram(profile, title, out_dir, filename):
    os.makedirs(out_dir, exist_ok=True)
    Z               = linkage(profile.values, method="ward")
    variables       = profile.columns.tolist()
    cluster_vectors = {i: profile.iloc[i].values for i in range(len(profile))}

    plt.figure(figsize=(10, 5))
    dendrogram(Z, labels=profile.index.astype(str))
    plt.gca().grid(False)
    plt.title(title)

    for i, (c1, c2, dist, _) in enumerate(Z):
        c1, c2  = int(c1), int(c2)
        v1, v2  = cluster_vectors[c1], cluster_vectors[c2]
        top_var = variables[np.argmax(np.abs(v1 - v2))]
        cluster_vectors[len(cluster_vectors)] = (v1 + v2) / 2
        plt.text(i + 1, dist, top_var, rotation=45, fontsize=8, va="bottom", ha="center")

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, filename), dpi=150, facecolor="white")
    plt.close()
    log.info(f"Dendrogram saved.")


# ==============================================================================
# 9. VISUALISATION — UMAP & t-SNE
# ==============================================================================

def build_cluster_palette(labels: np.ndarray) -> dict:
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])
    n_clusters      = len(unique_clusters)
    palette         = sns.color_palette("tab20", n_clusters) if n_clusters <= 20 \
                      else sns.color_palette("hsv", n_clusters)
    color_map       = {c: palette[i] for i, c in enumerate(unique_clusters)}
    color_map[-1]   = "lightgrey"
    return color_map


def plot_umap_2d(X, labels, title, out_dir, filename, metric="precomputed"):
    os.makedirs(out_dir, exist_ok=True)
    emb             = umap.UMAP(n_components=2, n_neighbors=VIZ_N_NEIGHBORS,
                                min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    color_map       = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])

    plt.figure(figsize=(9, 7))
    if (labels == -1).any():
        plt.scatter(emb[labels == -1, 0], emb[labels == -1, 1],
                    c="lightgrey", s=8, linewidth=0, label="Outliers", zorder=1, alpha=0.5)
    for c in unique_clusters:
        mask = labels == c
        plt.scatter(emb[mask, 0], emb[mask, 1], color=color_map[c],
                    s=10, linewidth=0, label=f"C{c}", zorder=2, alpha=0.8)

    plt.title(title)
    plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1),
               loc="upper left", markerscale=2, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, filename), dpi=150, facecolor="white")
    plt.close()
    log.info("UMAP 2D saved.")
    return color_map


def plot_umap_3d_html(X, labels, title, out_dir, filename_html,
                      metric="precomputed", color_map=None):
    os.makedirs(out_dir, exist_ok=True)
    emb             = umap.UMAP(n_components=3, n_neighbors=VIZ_N_NEIGHBORS,
                                min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])

    def to_hex(c):
        import matplotlib.colors as mcolors
        return "#d3d3d3" if c == "lightgrey" else mcolors.to_hex(c)

    df_plot = pd.DataFrame({"x": emb[:, 0], "y": emb[:, 1], "z": emb[:, 2],
                            "label": [str(l) for l in labels]})
    fig = px.scatter_3d(df_plot, x="x", y="y", z="z", color="label",
                        color_discrete_map={str(c): to_hex(color_map[c])
                                            for c in list(unique_clusters) + [-1]},
                        title=title, opacity=0.8,
                        category_orders={"label": ["-1"] + [str(c) for c in unique_clusters]})
    fig.update_traces(marker=dict(size=3))
    fig.write_html(os.path.join(out_dir, filename_html), include_plotlyjs="cdn")
    log.info("UMAP 3D saved.")


def plot_tsne_2d(X, labels, title, out_dir, filename,
                 metric="precomputed", color_map=None):
    os.makedirs(out_dir, exist_ok=True)
    init = umap.UMAP(n_components=2, n_neighbors=VIZ_N_NEIGHBORS,
                     min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    emb  = TSNE(n_components=2, perplexity=VIZ_TSNE_PERP, learning_rate="auto",
                init=init, metric=metric, max_iter=VIZ_TSNE_ITER).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])

    plt.figure(figsize=(9, 7))
    if (labels == -1).any():
        plt.scatter(emb[labels == -1, 0], emb[labels == -1, 1],
                    c="lightgrey", s=8, linewidth=0, label="Outliers", zorder=1, alpha=0.5)
    for c in unique_clusters:
        mask = labels == c
        plt.scatter(emb[mask, 0], emb[mask, 1], color=color_map[c],
                    s=10, linewidth=0, label=f"C{c}", zorder=2, alpha=0.8)
    plt.title(title)
    plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1),
               loc="upper left", markerscale=2, fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, filename), dpi=150, facecolor="white")
    plt.close()
    log.info("t-SNE 2D saved.")


def plot_tsne_3d_html(X, labels, title, out_dir, filename_html,
                      metric="precomputed", color_map=None):
    os.makedirs(out_dir, exist_ok=True)
    init = umap.UMAP(n_components=3, n_neighbors=VIZ_N_NEIGHBORS,
                     min_dist=VIZ_MIN_DIST, metric=metric).fit_transform(X)
    emb  = TSNE(n_components=3, perplexity=VIZ_TSNE_PERP, learning_rate="auto",
                init=init, metric=metric, max_iter=VIZ_TSNE_ITER).fit_transform(X)
    if color_map is None:
        color_map = build_cluster_palette(labels)
    unique_clusters = sorted([c for c in np.unique(labels) if c != -1])

    def to_hex(c):
        import matplotlib.colors as mcolors
        return "#d3d3d3" if c == "lightgrey" else mcolors.to_hex(c)

    df_plot = pd.DataFrame({"x": emb[:, 0], "y": emb[:, 1], "z": emb[:, 2],
                            "label": [str(l) for l in labels]})
    fig = px.scatter_3d(df_plot, x="x", y="y", z="z", color="label",
                        color_discrete_map={str(c): to_hex(color_map[c])
                                            for c in list(unique_clusters) + [-1]},
                        title=title, opacity=0.8,
                        category_orders={"label": ["-1"] + [str(c) for c in unique_clusters]})
    fig.update_traces(marker=dict(size=3))
    fig.write_html(os.path.join(out_dir, filename_html), include_plotlyjs="cdn")
    log.info("t-SNE 3D saved.")


# ==============================================================================
# 10. OUTLIER DESCRIPTION
# ==============================================================================

def describe_outliers_internal(df_sub, labels, run_label, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    df_work              = df_sub.copy()
    df_work["cluster"]   = labels
    df_noise             = df_work[df_work["cluster"] == -1]
    df_clustered         = df_work[df_work["cluster"] != -1]
    n_noise, n_total     = len(df_noise), len(df_work)

    log.info(f"[{run_label}] Outliers : {n_noise} / {n_total} ({100*n_noise/n_total:.1f}%)")
    if n_noise == 0:
        log.info("No outliers.")
        return

    numeric_cols = list(df_sub.select_dtypes(include="number").columns)
    compare      = pd.DataFrame({
        "outliers":  df_noise[numeric_cols].mean().round(3),
        "clustered": df_clustered[numeric_cols].mean().round(3),
    })
    compare["diff"] = (compare["outliers"] - compare["clustered"]).round(3)
    compare = compare.sort_values("diff", key=abs, ascending=False)

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.8), 4))
    sns.heatmap(compare[["outliers", "clustered"]].T, annot=True, fmt=".2f",
                cmap="YlOrRd", ax=ax, annot_kws={"size": 8})
    ax.set_title(f"Outliers vs clustered — {run_label}")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "outliers_internal_heatmap.png"),
                dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.8), 5))
    colors = ["tab:red" if v > 0 else "tab:blue" for v in compare["diff"]]
    ax.bar(compare.index, compare["diff"], color=colors, alpha=0.8)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylabel("Difference (outliers - clustered)")
    ax.set_title(f"Outliers vs clustered differences — {run_label}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "outliers_internal_diff.png"),
                dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()

    compare.to_csv(os.path.join(out_dir, "outliers_internal_compare.csv"))
    log.info(f"[{run_label}] Outlier description done.")


# ==============================================================================
# 11. POSTPROCESSING
# ==============================================================================

def run_postprocessing(df_sub, D, labels, run_label, out_dir):
    t_start = time.time()

    df_sweep = run_hdbscan_sweep(D)
    df_sweep = compute_combined_score(df_sweep)
    df_sweep.drop(columns=["labels", "probabilities", "cluster_sizes"]) \
            .to_csv(os.path.join(out_dir, "sweep.csv"), index=False)
    plot_sweep_curves(df_sweep, title=f"Sweep — {run_label}",
                      out_dir=out_dir, filename_prefix=run_label)

    profile = plot_cluster_heatmap(df_sub, labels,
                                   title=f"Cluster profiles — {run_label}",
                                   out_dir=out_dir, filename="heatmap.png")
    if len(profile) >= 2:
        plot_cluster_dendrogram(profile, title=f"Dendrogram — {run_label}",
                                out_dir=out_dir, filename="dendrogram.png")

    color_map = plot_umap_2d(D, labels, title=f"UMAP 2D — {run_label}",
                             out_dir=out_dir, filename="umap2d.png")
    plot_umap_3d_html(D, labels, title=f"UMAP 3D — {run_label}",
                      out_dir=out_dir, filename_html="umap3d.html", color_map=color_map)
    plot_tsne_2d(D, labels, title=f"t-SNE 2D — {run_label}",
                 out_dir=out_dir, filename="tsne2d.png", color_map=color_map)
    plot_tsne_3d_html(D, labels, title=f"t-SNE 3D — {run_label}",
                      out_dir=out_dir, filename_html="tsne3d.html", color_map=color_map)

    describe_outliers_internal(df_sub, labels, run_label,
                               out_dir=os.path.join(out_dir, "outliers_internal"))

    log.info(f"[{run_label}] Total postprocessing : {time.time()-t_start:.1f}s")


# ==============================================================================
# 12. ENTRY POINT
# ==============================================================================

if __name__ == "__main__":
    df = load_data(CSV_PATH)

    run_label = "s2_noweights_boolonly"
    out_dir   = os.path.join(OUTPUT_DIR, run_label)

    df_sub, D, labels, clusterer = run_hdbscan(
        df,
        run_label        = run_label,
        min_cluster_size = HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples      = HDBSCAN_MIN_SAMPLES,
    )

    run_postprocessing(df_sub, D, labels, run_label, out_dir)

GPU : NVIDIA A100-SXM4-80GB
Available memory : 85 GB


INFO | [s2_noweights_boolonly] binary cols = 12 | n = 29839


Dataset loaded : 29839 rows, 179 columns


INFO | [s2_noweights_boolonly] Distance matrix : 45.9s
INFO | [s2_noweights_boolonly] HDBSCAN : 72.6s
INFO | [s2_noweights_boolonly] clusters=8 | noise=2747 (9.2%)
INFO | [s2_noweights_boolonly] CSV exported.
INFO | [s2_noweights_boolonly] Sweep curves saved.
INFO | Heatmap saved.
INFO | Dendrogram saved.
/home/nadia/pycharm_project_nad/.venv/lib/python3.12/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
INFO | UMAP 2D saved.
/home/nadia/pycharm_project_nad/.venv/lib/python3.12/site-packages/umap/umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
INFO | UMAP 3D saved.
/home/nadia/pycharm_project_nad/.venv/lib/python3.12/site-packages/umap/umap_.py:1865: UserWarning:

using precomputed metric; inverse_transform will be unavailable

INFO | 

"Plusieurs configurations ont été testées — booléens seuls, MinMaxScaler, StandardScaler. La configuration avec StandardScaler et counts a produit les clusters les plus stables et les mieux séparés, suggérant que l'intensité de consommation est le principal facteur différenciant les patients, davantage que le type d'examen spécifique."